# LoRA — Paper-to-Code Mock (Colab)

**Paper:** LoRA: Low-Rank Adaptation of Large Language Models (Hu et al., 2021) — https://arxiv.org/abs/2106.09685

Timebox: ~40 min. Fill in the `LoRALinear` stub, train on the toy task, then run the sanity checks. Reference solution is in the last cell — don't peek until you've tried.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## 1. Implement the LoRA layer (YOUR TASK)

Forward:  `h = x W0^T + (alpha/r) * x A^T B^T`, with `W0` frozen and only `A`, `B` trainable.

Remember: `A` is random, `B` is **zero** at init, and project down to rank `r` first.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, r=4, alpha=8, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.r = r
        self.scaling = alpha / r

        # --- frozen base layer ---
        self.base = nn.Linear(in_features, out_features, bias=bias)
        # TODO: freeze base.weight (and base.bias if present)

        # --- trainable low-rank update ---
        # TODO: self.A = nn.Parameter(...)  shape (r, in), random
        # TODO: self.B = nn.Parameter(...)  shape (out, r), zeros
        raise NotImplementedError("fill me in")

    def forward(self, x):
        # TODO: base path + scaling * low-rank path
        raise NotImplementedError("fill me in")

    @torch.no_grad()
    def merged_weight(self):
        # TODO: return base.weight + scaling * (B @ A)
        raise NotImplementedError("fill me in")

## 2. Toy training task
Recover a random linear map. Loss should drop steadily.

In [ ]:
in_dim, out_dim = 64, 32
layer = LoRALinear(in_dim, out_dim, r=4, alpha=8)
true_W = torch.randn(out_dim, in_dim)

base_snapshot = layer.base.weight.clone()  # for sanity check 3
opt = torch.optim.Adam([p for p in layer.parameters() if p.requires_grad], lr=1e-2)

for step in range(300):
    x = torch.randn(128, in_dim)
    y = x @ true_W.T
    loss = F.mse_loss(layer(x), y)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0:
        print(f"step {step:3d}  loss {loss.item():.4f}")

## 3. Sanity checks

In [ ]:
# 1) only A and B are trainable
trainable = [n for n, p in layer.named_parameters() if p.requires_grad]
n_train = sum(p.numel() for p in layer.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in layer.parameters())
print("trainable:", trainable, f"({n_train}/{n_total} = {100*n_train/n_total:.1f}%)")

# 2) init: B=0 so layer == base
fresh = LoRALinear(in_dim, out_dim, r=4, alpha=8)
xb = torch.randn(10, in_dim)
assert torch.allclose(fresh(xb), fresh.base(xb), atol=1e-6)

# 3) base weight unchanged by training
assert torch.equal(layer.base.weight, base_snapshot)

# 4) shapes
assert layer.A.shape == (4, in_dim) and layer.B.shape == (out_dim, 4)

# 5) merged weight reproduces output (zero-latency claim)
xt = torch.randn(16, in_dim)
out_merged = xt @ layer.merged_weight().T + layer.base.bias
assert torch.allclose(layer(xt), out_merged, atol=1e-5)

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
class LoRALinearSolution(nn.Module):
    def __init__(self, in_features, out_features, r=4, alpha=8, bias=True):
        super().__init__()
        self.r = r
        self.scaling = alpha / r
        self.base = nn.Linear(in_features, out_features, bias=bias)
        self.base.weight.requires_grad_(False)
        if bias:
            self.base.bias.requires_grad_(False)
        self.A = nn.Parameter(torch.randn(r, in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_features, r))

    def forward(self, x):
        return self.base(x) + self.scaling * ((x @ self.A.T) @ self.B.T)

    @torch.no_grad()
    def merged_weight(self):
        return self.base.weight + self.scaling * (self.B @ self.A)